# Evaluation:


## S0: Configuration

In [ ]:
BASE        = "unsloth/Llama-3.2-1B"
CPT_ADAPTER = "/kaggle/input/notebooks/tillkokemoor/cpt-notebook/cpt_merged"
SFT_MODEL   = "/kaggle/input/notebooks/tillkokemoor/sft-notebook/sft_output"
MAX_SEQ_LENGTH = 1024

## S1: Installation

In [ ]:
!pip install unsloth unsloth_zoo bitsandbytes rouge_score

## S2: Eval-Set

In [ ]:
EVAL = [
    # ---- NEAR ----
    {"q": "Why can't AUTOSAR software components communicate directly without the RTE?",
     "ref": "The RTE is the middleware that decouples SWCs from each other and from the BSW; without it there is no standardized communication path.",
     "type": "near"},
    {"q": "If two ECUs start transmitting on the CAN bus at the same time, which one wins and why?",
     "ref": "CAN uses bitwise arbitration (CSMA/CR): the message with the lower identifier (more dominant bits) wins non-destructively; the loser backs off and retransmits later.",
     "type": "near"},
    {"q": "What is the difference between ASIL-A and ASIL-D?",
     "ref": "ASIL (ISO 26262) ranges from A (lowest) to D (highest integrity). ASIL-D covers the highest-risk hazards and demands the strictest safety measures; ASIL-A the least.",
     "type": "near"},
    {"q": "Why does CAN-FD achieve higher throughput than classical CAN?",
     "ref": "CAN-FD allows a higher bit rate in the data phase and larger payloads (up to 64 bytes vs. 8), so more data is sent per frame and faster.",
     "type": "near"},
    # ---- FAR ----
    {"q": "What is FlexRay and how does its TDMA scheduling work?",
     "ref": "FlexRay is a deterministic time-triggered automotive bus (up to 10 Mbit/s) for safety-critical systems. A static segment assigns fixed TDMA slots per node; a dynamic segment carries event-driven messages.",
     "type": "far"},
    {"q": "What is Time-Sensitive Networking (TSN) in automotive Ethernet?",
     "ref": "TSN is a set of IEEE 802.1 standards adding deterministic, low-latency delivery to standard Ethernet via time synchronization, time-aware scheduling, and frame preemption.",
     "type": "far"},
    {"q": "How does UDS Service 0x2E WriteDataByIdentifier work?",
     "ref": "A UDS (ISO 14229) diagnostic service that writes data to a record addressed by a 2-byte Data Identifier (DID): request 0x2E + DID + data, positive response 0x6E + DID.",
     "type": "far"},
    {"q": "What does the WdgM module do?",
     "ref": "WdgM (AUTOSAR Watchdog Manager) supervises program execution (alive, deadline, logical supervision) and triggers a reaction such as a watchdog reset on failure. It is unrelated to CAN.",
     "type": "far"},
]


## S3: Helpers

In [ ]:
from unsloth import FastLanguageModel
import torch

def run_stage(path, is_adapter=False):
    model, tok = FastLanguageModel.from_pretrained(
        model_name=path, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=False)
    if is_adapter:
        model = model.merge_and_unload()
    FastLanguageModel.for_inference(model)
    outs = []
    for ex in EVAL:
        prompt = f"### Instruction:\n{ex['q']}\n\n### Response:\n"
        ids = tok(prompt, return_tensors="pt").to("cuda")
        gen = model.generate(**ids, max_new_tokens=200, do_sample=False)
        outs.append(tok.decode(gen[0], skip_special_tokens=True)
                       .split("### Response:\n")[-1].strip())
    del model; torch.cuda.empty_cache()
    return outs

## S4:

In [ ]:
base_out = run_stage(BASE)
cpt_out  = run_stage(CPT_ADAPTER, is_adapter=True)
sft_out  = run_stage(SFT_MODEL)

## S5: ROUGE

In [ ]:
from rouge_score import rouge_scorer
import pandas as pd

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rl = lambda ref, out: scorer.score(ref, out)["rougeL"].fmeasure

rows = [{"type": e["type"], "q": e["q"][:35],
         "base": round(rl(e["ref"], b), 3),
         "cpt":  round(rl(e["ref"], c), 3),
         "cpt+sft": round(rl(e["ref"], s), 3)}
        for e, b, c, s in zip(EVAL, base_out, cpt_out, sft_out)]
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nMean:", df[["base","cpt","cpt+sft"]].mean().round(3).to_dict())

## S6:

In [ ]:
for e, b, c, s in zip(EVAL, base_out, cpt_out, sft_out):
    print(f"\n{'='*70}\n[{e['type'].upper()}] {e['q']}")
    print(f"REF:     {e['ref']}")
    print(f"BASE:    {b[:300]}")
    print(f"CPT:     {c[:300]}")
    print(f"CPT+SFT: {s[:300]}")